# Customer Churn Prediction and Retention Analysis

**Author:** Bhupender Sejwal

## Project Objective
This project analyzes telecom customer data to predict churn and identify the main business factors associated with customer attrition. The goal is to support retention planning by building classification models and translating the results into actionable business recommendations.

## 1. Business Problem

Customer churn is a major challenge for subscription-based businesses because losing customers reduces revenue and increases customer acquisition costs. In this project, the objective is to:

- predict whether a customer is likely to churn
- identify the variables most strongly associated with churn
- compare baseline and improved classification models
- provide business recommendations based on model output and exploratory analysis

Because churn is a minority class in this dataset, recall is treated as an important evaluation metric in addition to accuracy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

plt.rcParams["figure.figsize"] = (8, 5)
sns.set_style("whitegrid")

## 2. Data Loading

The dataset used in this analysis is the IBM Telco Customer Churn dataset. The notebook loads the CSV file from Google Drive so that the file remains available across Colab sessions.

In [ ]:
drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/churn.csv'
df = pd.read_csv(file_path)

df.head()

In [ ]:
print("Dataset shape:", df.shape)
df.info()

## 3. Dataset Overview

The dataset contains customer demographic details, subscription information, account attributes, payment details, and the target variable **Churn**. Each row represents one customer record.

In [ ]:
df.describe(include='all').transpose().head(10)

## 4. Data Cleaning and Preparation

Before modeling, the dataset must be cleaned and standardized. The following steps are performed:

- convert `TotalCharges` to numeric format
- remove rows with missing values
- drop `customerID` because it is an identifier and not useful for prediction
- convert the target variable `Churn` into binary format

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
df.drop('customerID', axis=1, inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("Cleaned dataset shape:", df.shape)
df.head()

In [ ]:
df.isnull().sum()

## 5. Class Distribution

This step checks whether the target variable is imbalanced. That is important because imbalanced classes can reduce the model's ability to detect churn correctly.

In [ ]:
churn_distribution = df['Churn'].value_counts()
churn_percentage = df['Churn'].value_counts(normalize=True) * 100

print("Class counts:")
print(churn_distribution)
print("\nClass percentages:")
print(churn_percentage.round(2))

In [ ]:
sns.countplot(x='Churn', data=df)
plt.title("Churn Distribution")
plt.xticks([0, 1], ['No Churn', 'Churn'])
plt.show()

## 6. Exploratory Data Analysis

The objective of EDA is to identify patterns associated with churn and support business interpretation before model training.

In [ ]:
contract_plot_df = df.copy()
contract_plot_df['Churn_Label'] = contract_plot_df['Churn'].map({0: 'No', 1: 'Yes'})

sns.countplot(x='Contract', hue='Churn_Label', data=contract_plot_df)
plt.title("Churn by Contract Type")
plt.xticks(rotation=20)
plt.show()

In [ ]:
billing_plot_df = df.copy()
billing_plot_df['Churn_Label'] = billing_plot_df['Churn'].map({0: 'No', 1: 'Yes'})

sns.boxplot(x='Churn_Label', y='MonthlyCharges', data=billing_plot_df)
plt.title("Monthly Charges vs Churn")
plt.show()

In [ ]:
tenure_plot_df = df.copy()
tenure_plot_df['Churn_Label'] = tenure_plot_df['Churn'].map({0: 'No', 1: 'Yes'})

sns.boxplot(x='Churn_Label', y='tenure', data=tenure_plot_df)
plt.title("Tenure vs Churn")
plt.show()

### EDA Observations

- Customers on month-to-month contracts show substantially higher churn than customers on one-year or two-year contracts.
- Customers with higher monthly charges are more likely to churn.
- Customers with lower tenure are more likely to churn, suggesting that newer customers are at greater risk of leaving.

These patterns indicate that contract structure, pricing, and customer lifetime stage are important business drivers of churn.

## 7. Feature Engineering and Preprocessing

To prepare the data for modeling:

- features and target are separated
- categorical variables are transformed using one-hot encoding
- the train-test split uses stratification to preserve the churn ratio
- numerical variables are scaled for Logistic Regression

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

X_train_lr = X_train.copy()
X_test_lr = X_test.copy()

scaler = StandardScaler()
X_train_lr[numeric_cols] = scaler.fit_transform(X_train_lr[numeric_cols])
X_test_lr[numeric_cols] = scaler.transform(X_test_lr[numeric_cols])

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

## 8. Model 1: Logistic Regression (Balanced)

A class-balanced Logistic Regression model is used to improve churn detection. This is important because missing actual churn cases can be more costly to the business than incorrectly flagging a non-churn customer.

In [ ]:
lr_model = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
lr_model.fit(X_train_lr, y_train)

y_pred_lr = lr_model.predict(X_test_lr)

print("Logistic Regression (Balanced) Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_lr))

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)

sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Logistic Regression (Balanced)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

### Logistic Regression Interpretation

This model prioritizes recall for churn detection. In a churn use case, identifying customers who are likely to leave is more valuable than optimizing only for raw accuracy.

## 9. Model 2: Random Forest

A Random Forest classifier is trained to capture non-linear relationships in the data and provide feature importance scores for interpretation.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf))

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)

sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens')
plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 10. Model Comparison

The following table compares the two models using accuracy, precision, recall, and F1 score for the churn class.

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression (Balanced)', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf)
    ],
    'Precision (Churn)': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf)
    ],
    'Recall (Churn)': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf)
    ],
    'F1 Score (Churn)': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf)
    ]
})

results.sort_values(by='Recall (Churn)', ascending=False)

### Model Comparison Interpretation

The balanced Logistic Regression model is especially useful when the objective is to capture more churn cases. Random Forest provides additional predictive strength and interpretability through feature importance. The final model choice depends on whether the business prioritizes higher recall or a more balanced performance profile.

## 11. Feature Importance Analysis

Random Forest feature importance helps identify the variables that contribute most to churn prediction.

In [ ]:
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

importance_df.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(10))
plt.title("Top 10 Important Features for Churn Prediction")
plt.show()

### Feature Importance Insights

The most influential variables generally include billing-related features, tenure, and contract information. This indicates that churn behavior is strongly linked to pricing pressure, customer loyalty duration, and contract commitment.

## 12. Business Recommendations

Based on the exploratory analysis and model results, the following recommendations are suggested:

1. Prioritize retention campaigns for customers on month-to-month contracts.
2. Monitor customers with high monthly charges and consider personalized retention offers.
3. Focus onboarding and engagement strategies on newer customers with low tenure.
4. Review customer experience and payment friction for higher-risk payment methods.
5. Use churn scores to support proactive retention outreach before customers leave.

## 13. Conclusion

This project demonstrated an end-to-end machine learning workflow for customer churn prediction using a telecom dataset with more than 7,000 records. The analysis included data cleaning, exploratory analysis, preprocessing, classification modeling, model comparison, and feature importance interpretation.

The project shows that churn can be meaningfully predicted using customer billing, tenure, and contract-related variables. It also demonstrates how model evaluation should be aligned with business priorities, especially when recall is more valuable than raw accuracy.